# Cross-Industry Accelerator — Configuration
### Universal Setup for All Industry Implementations

This notebook defines the **industry configuration** used by all downstream notebooks.
Run this notebook first, then `%run` it from other notebooks to import the config.

**Supported Industries:** Healthcare, Finance, Insurance, Retail, CPG, Construction, Oil & Gas, Media, Law Firms, Telecom

---

### How It Works
1. Set `INDUSTRY` to your target industry key
2. The notebook auto-discovers CSV files from the data folder
3. Files are classified as `dim_*`, `fact_*`, or `stream_*` based on naming convention
4. All downstream notebooks inherit this configuration

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 1: SELECT YOUR INDUSTRY
# ════════════════════════════════════════════════════════════════════════

INDUSTRY = "healthcare"  # Change this to your target industry

# Supported values:
# "healthcare", "finance", "insurance", "retail", "cpg",
# "construction", "oil_and_gas", "media", "law_firms", "telecom"

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2: FABRIC ARTIFACT NAMES
# ════════════════════════════════════════════════════════════════════════
# These follow the naming convention from INDUSTRY_REPLICATION_GUIDE.md
# Override any value below if your workspace uses different names.

INDUSTRY_DISPLAY_NAMES = {
    "healthcare":   "Healthcare",
    "finance":      "Finance",
    "insurance":    "Insurance",
    "retail":       "Retail",
    "cpg":          "CPG",
    "construction": "Construction",
    "oil_and_gas":  "OilGas",
    "media":        "Media",
    "law_firms":    "LawFirm",
    "telecom":      "Telecom",
}

INDUSTRY_LABEL = INDUSTRY_DISPLAY_NAMES.get(INDUSTRY, INDUSTRY.title())

# ── Fabric Artifact Names ──────────────────────────────────────────────
LAKEHOUSE_NAME     = f"{INDUSTRY_LABEL}_Data_Bronze"
WAREHOUSE_NAME     = f"{INDUSTRY_LABEL}_Datawarehouse"
EVENTHOUSE_NAME    = f"{INDUSTRY.replace(' ', '_')}_rt_store"
ONTOLOGY_NAME      = f"{INDUSTRY_LABEL}DocBurdenOntology"
DATA_AGENT_NAME    = f"{INDUSTRY_LABEL}QA"
OPS_AGENT_NAME     = f"{INDUSTRY_LABEL}DocBurden"
SEMANTIC_MODEL_NAME = f"{INDUSTRY_LABEL}_DocBurden_Model"

print(f"Industry:       {INDUSTRY} ({INDUSTRY_LABEL})")
print(f"Lakehouse:      {LAKEHOUSE_NAME}")
print(f"Warehouse:      {WAREHOUSE_NAME}")
print(f"Eventhouse:     {EVENTHOUSE_NAME}")
print(f"Ontology:       {ONTOLOGY_NAME}")
print(f"Data Agent:     {DATA_AGENT_NAME}")
print(f"Ops Agent:      {OPS_AGENT_NAME}")
print(f"Semantic Model: {SEMANTIC_MODEL_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 3: DATA PATHS & BINDING CONFIGURATION
# ════════════════════════════════════════════════════════════════════════

# Path where CSV files are uploaded in the Lakehouse Files area
CSV_BASE_PATH = f"/lakehouse/default/Files/{INDUSTRY}_data"

# Lakehouse schema (use 'dbo' for schema-enabled lakehouses)
LAKEHOUSE_SCHEMA = "dbo"

# Warehouse schema
WAREHOUSE_SCHEMA = "dbo"

# Eventhouse connection (fill these in from your Fabric workspace)
EVENTHOUSE_CLUSTER_URI = ""   # e.g. https://<name>.<region>.kusto.fabric.microsoft.com
EVENTHOUSE_DATABASE     = ""   # your Eventhouse DB name

# Ontology package path (if using .iq package)
ONTOLOGY_PACKAGE_PATH = f"/lakehouse/default/Files/{INDUSTRY}_ontology_package.iq"

print(f"CSV source:    {CSV_BASE_PATH}")
print(f"LH schema:     {LAKEHOUSE_SCHEMA}")
print(f"WH schema:     {WAREHOUSE_SCHEMA}")
print(f"Eventhouse:    {EVENTHOUSE_CLUSTER_URI or '(not configured)'}")
print(f"Ontology pkg:  {ONTOLOGY_PACKAGE_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 4: AUTO-DISCOVER DATA SOURCES
# ════════════════════════════════════════════════════════════════════════
# Scans the CSV folder and classifies tables by naming convention:
#   dim_*    → Dimension tables  → Lakehouse + Warehouse
#   fact_*   → Fact tables       → Lakehouse + Warehouse (batch) or Eventhouse (events)
#   stream_* → Streaming tables  → Eventhouse only

import os
import json

def discover_data_sources(base_path):
    """Scan CSV directory and classify files by naming convention."""
    dim_tables = []
    fact_tables_batch = []    # fact tables for Lakehouse/Warehouse
    fact_tables_event = []    # fact tables for Eventhouse (event-level)
    stream_tables = []        # stream tables for Eventhouse only

    try:
        files = [f for f in os.listdir(base_path) if f.endswith('.csv')]
    except FileNotFoundError:
        print(f"ERROR: Path not found: {base_path}")
        print("Upload your CSV files to the Lakehouse Files area first.")
        return dim_tables, fact_tables_batch, fact_tables_event, stream_tables

    for f in sorted(files):
        table_name = f.replace('.csv', '')
        if table_name.startswith('dim_'):
            dim_tables.append(table_name)
        elif table_name.startswith('stream_'):
            stream_tables.append(table_name)
        elif table_name.startswith('fact_'):
            # Heuristic: event-level facts contain time-series / clickstream data
            # These keywords indicate event tables → Eventhouse
            event_keywords = [
                '_events', '_interactions', '_clickstream', '_scans',
                '_alerts', '_location', '_handoff', '_administration',
                '_vital', '_assessments', '_escalation', '_transfers',
                '_integrity', '_inspections', '_rfi_', '_change_order',
                '_filing', '_contract', '_metadata', '_approval',
                '_delivery', '_regulatory', '_outage', '_rca',
                '_dispatch', '_nms_', '_dms_', '_mam_',
            ]
            if any(kw in table_name for kw in event_keywords):
                fact_tables_event.append(table_name)
            else:
                fact_tables_batch.append(table_name)
        else:
            # Unknown prefix — treat as batch fact
            fact_tables_batch.append(table_name)

    return dim_tables, fact_tables_batch, fact_tables_event, stream_tables

# Run discovery
DIM_TABLES, FACT_TABLES_BATCH, FACT_TABLES_EVENT, STREAM_TABLES = discover_data_sources(CSV_BASE_PATH)

# Combined views for different targets
LAKEHOUSE_TABLES = DIM_TABLES + FACT_TABLES_BATCH   # All dims + batch facts → Lakehouse
WAREHOUSE_TABLES = DIM_TABLES + FACT_TABLES_BATCH + FACT_TABLES_EVENT  # All non-streaming → Warehouse
EVENTHOUSE_TABLES = FACT_TABLES_EVENT + STREAM_TABLES  # Events + streams → Eventhouse

print(f"\n{'='*60}")
print(f"DATA SOURCE DISCOVERY — {INDUSTRY.upper()}")
print(f"{'='*60}")
print(f"\nDimension tables ({len(DIM_TABLES)}):")
for t in DIM_TABLES: print(f"  • {t}")
print(f"\nFact tables — Batch/Lakehouse ({len(FACT_TABLES_BATCH)}):")
for t in FACT_TABLES_BATCH: print(f"  • {t}")
print(f"\nFact tables — Event/Eventhouse ({len(FACT_TABLES_EVENT)}):")
for t in FACT_TABLES_EVENT: print(f"  • {t}")
print(f"\nStreaming tables — Eventhouse ({len(STREAM_TABLES)}):")
for t in STREAM_TABLES: print(f"  • {t}")
print(f"\n{'─'*60}")
print(f"Lakehouse target: {len(LAKEHOUSE_TABLES)} tables")
print(f"Warehouse target: {len(WAREHOUSE_TABLES)} tables")
print(f"Eventhouse target: {len(EVENTHOUSE_TABLES)} tables")
print(f"Total CSV files:  {len(DIM_TABLES) + len(FACT_TABLES_BATCH) + len(FACT_TABLES_EVENT) + len(STREAM_TABLES)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 5: SCHEMA INSPECTION (preview any table)
# ════════════════════════════════════════════════════════════════════════

def preview_table(table_name, base_path=CSV_BASE_PATH, rows=5):
    """Read a CSV and display schema + sample rows."""
    path = f"{base_path}/{table_name}.csv"
    df = spark.read.option("header", True).option("inferSchema", True).csv(path)
    print(f"\n{'─'*60}")
    print(f"Table: {table_name}  |  {df.count()} rows  |  {len(df.columns)} columns")
    print(f"{'─'*60}")
    df.printSchema()
    df.show(rows, truncate=False)
    return df

# Preview the first dimension and first fact table
if DIM_TABLES:
    preview_table(DIM_TABLES[0])
if FACT_TABLES_BATCH:
    preview_table(FACT_TABLES_BATCH[0])

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIG SUMMARY (exported to downstream notebooks via %run)
# ════════════════════════════════════════════════════════════════════════

CONFIG = {
    "industry":             INDUSTRY,
    "industry_label":       INDUSTRY_LABEL,
    "lakehouse_name":       LAKEHOUSE_NAME,
    "warehouse_name":       WAREHOUSE_NAME,
    "eventhouse_name":      EVENTHOUSE_NAME,
    "eventhouse_uri":       EVENTHOUSE_CLUSTER_URI,
    "eventhouse_db":        EVENTHOUSE_DATABASE,
    "ontology_name":        ONTOLOGY_NAME,
    "ontology_package":     ONTOLOGY_PACKAGE_PATH,
    "data_agent_name":      DATA_AGENT_NAME,
    "ops_agent_name":       OPS_AGENT_NAME,
    "semantic_model_name":  SEMANTIC_MODEL_NAME,
    "csv_base_path":        CSV_BASE_PATH,
    "lakehouse_schema":     LAKEHOUSE_SCHEMA,
    "warehouse_schema":     WAREHOUSE_SCHEMA,
    "dim_tables":           DIM_TABLES,
    "fact_tables_batch":    FACT_TABLES_BATCH,
    "fact_tables_event":    FACT_TABLES_EVENT,
    "stream_tables":        STREAM_TABLES,
    "lakehouse_tables":     LAKEHOUSE_TABLES,
    "warehouse_tables":     WAREHOUSE_TABLES,
    "eventhouse_tables":    EVENTHOUSE_TABLES,
}

print(json.dumps({k: v if not isinstance(v, list) else f"{len(v)} tables" for k, v in CONFIG.items()}, indent=2))